# Refresh — re-read saved videos / channels from the YouTube API (step 1 of the daily job)

Runs the app's `refresh_daily.py` from `src/app` (deployed with the bundle, so the job and the app share one copy of the code) on serverless. It writes the latest counters into Lakebase Postgres `public.videos` / `public.channels`, so Bronze → Silver get one snapshot per video / channel per day.

**Empty-source guard:** `refresh_daily.py` reads everything first and raises before writing if the API returns 0 rows or more than 20% zero counters, so this task fails and the rest of the job does not run (no day of zeros).

**Secrets (scope `youtube`):** `pg_password` (Postgres role `admin`), `api_key` (YouTube Data API v3).

Widget `dry_run` = `true` calls the API and runs the guard but writes nothing.

**Postgres driver on serverless:** serverless runs OpenSSL in FIPS mode. The OpenSSL bundled in the `psycopg[binary]` wheel fails the FIPS self-test and aborts the process (SIGABRT), so this notebook installs plain `psycopg` and sets `PSYCOPG_IMPL=python` (pure-Python implementation on the system libpq / OpenSSL).

`refresh_daily.py` runs as a **separate Python process** (same interpreter, same `%pip` packages), like it does locally. The subprocess gets the secrets through its environment, and its exit code (non-zero on the empty-source guard) decides whether this task fails.


In [ ]:
%pip install --quiet "psycopg>=3.2.0" "httpx>=0.28.0" "python-dotenv>=1.0.0"


In [ ]:
# --- Config ---
import os
import sys

dbutils.widgets.dropdown("dry_run", "false", ["false", "true"])
DRY_RUN = dbutils.widgets.get("dry_run") == "true"

refresh_env = {**os.environ, **{
    "PGHOST": "ep-still-butterfly-d8fd3r70.database.us-east-2.cloud.databricks.com",
    "PGDATABASE": "databricks_postgres",
    "PGUSER": "admin",
    "PGPORT": "5432",
    "PGSSLMODE": "require",
    "APP_SCHEMA": "public",
    "PGPASSWORD": dbutils.secrets.get("youtube", "pg_password"),
    "YOUTUBE_API_KEY": dbutils.secrets.get("youtube", "api_key"),
    # Serverless runs OpenSSL in FIPS mode: the OpenSSL bundled in psycopg[binary] fails the FIPS
    # self-test and aborts, so use psycopg's pure-Python implementation on the system libpq.
    "PSYCOPG_IMPL": "python",
}}

notebook_dir = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
app_dir = os.path.normpath(f"/Workspace{notebook_dir}/../../app")  # src/notebooks/refresh -> src/app
print(f"App code: {app_dir} · dry_run = {DRY_RUN}")


In [ ]:
# --- Refresh in its own process (exits non-zero on EmptySourceError, before anything is written) ---
import subprocess

result = subprocess.run(
    [sys.executable, "-X", "faulthandler", "refresh_daily.py", *(["--dry-run"] if DRY_RUN else [])],
    cwd=app_dir, env=refresh_env, capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr[-4000:])
    raise RuntimeError(f"refresh_daily.py failed (exit code {result.returncode}); nothing after the failing step was written\n{result.stderr[-3000:]}")
print("✅ Refresh complete")
